# Fit of an effective Hamiltonian on the Floquet spectrum of the ATS

1. One-period propagator of the closed (no dissipation) full-ATS Hamiltonian,
   integrated in the **interaction frame** of its static part,
   `floquet_lindblad.models.ats.build_ats_hamiltonian_interaction`, with
   `res = dq.sepropagator(H_I, tsave)`.
2. Its **Floquet spectrum**: diagonalize $U = \sum_j \lambda_j |v_j\rangle\langle v_j|$
   and take the log, $\epsilon_j = \frac{i}{T}\log\lambda_j = -\arg(\lambda_j)/T$.
3. A **least-squares fit** (`scipy.optimize.least_squares`) of the
   time-independent effective model

$$
\begin{aligned}
H_{\rm RWA}={}&(\delta_a+\Delta_a)\,a^\dagger a+(\delta_b+\Delta_b)\,b^\dagger b
+g_2\left(a^2b^\dagger+a^{\dagger2}b\right)+\frac{\varepsilon_d}{2}\left(b+b^\dagger\right)
+\varepsilon_2\left(a^{\dagger2}+a^2\right)\\
&-\frac{K_a}{2}\,a^{\dagger2}a^2-\frac{K_b}{2}\,b^{\dagger2}b^2-\chi\,a^\dagger a\,b^\dagger b
\end{aligned}
$$

   on that spectrum, jointly at several drive strengths, starting from the
   second-order RWA values of `notebooks/sympy_rwa_ats.ipynb` (as used in
   `RWA/notebooks/sweep_alpha_RWA_compare_terms.ipynb`).

**Interaction frame.** The builder splits $H(t) = H_s + H_d(t)$ (static part,
drive) and returns $H_I(t) = e^{iH_st}H_d(t)e^{-iH_st}$ written in the eigenbasis
$V$ of $H_s$, with `output_phase` $= e^{-i\lambda T}$ ($\lambda$: eigenvalues of
$H_s$). The one-period map is then

$$U = e^{-iH_sT}\,U_I(T) = {\rm diag}(\texttt{output\_phase})\,U_I(T)
\quad\text{(in the basis } V\text{)},$$

unitarily equivalent to the lab-frame propagator -- same quasienergies -- and its
eigenvectors are brought back to the Fock basis with $V$. Only $H_I$ is
integrated, so the ODE does not resolve the fast $e^{-iH_st}$: about 1 s per
period, and quasienergies accurate to ~$10^{-11}$ GHz (lab frame: 10-30 s,
~$10^{-7}$).

**Which period.** $T = 2\pi/\omega_a$, the builder's `T_block`: two drive
periods ($\omega_b = 2\omega_a$), over which the free evolution at
$(\omega_a, \omega_b)$ is the identity, so the quasienergies are those of the
frame rotating at $(\omega_a, \omega_b)$, where $H_{\rm RWA}$ lives; there
$\delta_a = \delta_b = 0$ (the builder drives exactly at $\omega_b = 2\omega_a$).
The spectrum is the right thing to fit: it does not depend on the Floquet gauge
(the time origin of the period), unlike the matrix elements of $\log U$, which
differ from the effective Hamiltonian by the micromotion, at the same order
$g^2/\omega$ as the corrections fitted here.

**Where it runs.** The propagators run on the anb-compute Ray cluster, in one
`acr.run`: one GPU task per drive and truncation, then one CPU task per drive that
sorts the converged Floquet states from the truncation artefacts. Only the
converged states come back to the notebook -- the full $N\times N$ eigenvector
matrices ($N = n_a n_b$, 1250 at `DIMS`) stay on the cluster. The fit runs
locally, on a fraction of them, and takes seconds: one diagonalization per
iteration (analytic Jacobian), split by storage parity, in a Fock box just large
enough for the states it fits.

**Kernel**: the workspace pixi environment (Python 3.13.x, `ray 2.56.0`), with a
Vault session open (`vault login -method=oidc role=research_core_user`), as for
`notebooks/sweeps/`. Restart the kernel if something already imported `ray`.

In [ ]:
import os
import pathlib
import subprocess
import sys
import time

assert sys.version_info[:2] == (3, 13), (
    f"kernel runs Python {sys.version.split()[0]}; the cluster needs 3.13.x. "
    "Select the workspace pixi environment as the kernel."
)

# Ray reads RAY_AUTH_MODE once, at its first import, and caches it. Setting it
# here works *only* because nothing has imported ray yet in a fresh kernel.
os.environ["RAY_AUTH_MODE"] = "token"
os.environ.setdefault("VAULT_ADDR", "https://vault.int.alice-bob.com")

if not os.environ.get("RAY_AUTH_TOKEN"):
    try:
        os.environ["RAY_AUTH_TOKEN"] = subprocess.run(
            ["vault", "kv", "get", "-field=token", "kv-it/prod/ray/auth-token"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        raise RuntimeError(
            "Could not mint the cluster token from Vault. In a terminal, run:\n"
            "  export VAULT_ADDR=https://vault.int.alice-bob.com\n"
            "  vault login -method=oidc role=research_core_user\n"
            "then re-run this cell (no kernel restart needed).\n"
            f"{(getattr(exc, 'stderr', '') or '').strip()}"
        ) from exc

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import least_squares, linear_sum_assignment

from anb_compute import ray as acr


def _find_repo():
    """Locate the repo checkout without hardcoding an absolute path.

    Notebooks have no __file__, so search upward from the working directory --
    which VS Code sets to either the notebook's folder or the workspace root.
    """
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "src" / "floquet_lindblad").is_dir():
            return base
        for hit in sorted(base.glob("*/src/floquet_lindblad")):
            return hit.parents[1]
    raise RuntimeError(f"no checkout with src/floquet_lindblad found above {here}")


REPO = _find_repo()

# Shipped by PATH, not by name (a non-editable install would ship a frozen
# copy). They prepend to the worker's sys.path, so this dynamiqs shadows the
# image's stock one. The client imports neither: everything that needs jax or
# dynamiqs runs on the workers.
PKG = str(REPO / "src" / "floquet_lindblad")
# The patched dynamiqs checkout, expected beside the repo; $DYNAMIQS_SRC wins.
DYNAMIQS = os.environ.get("DYNAMIQS_SRC") or str(REPO.parent / "dynamiqs" / "dynamiqs")
for _p in (PKG, DYNAMIQS):
    if not pathlib.Path(_p).is_dir():
        raise RuntimeError(f"not a directory, cannot ship to workers: {_p}")

RUNTIME_ENV = acr.build_runtime_env(py_modules=[PKG, DYNAMIQS])
print("ready")

## Settings

**Several drive strengths, fitted together.** With the drive on, the buffer of
the *closed* system (no $\kappa_b$) is driven close to resonance
($\varepsilon_d/2 = g\,\alpha^2$ against $|\Delta_b| \approx 10^{-2}$), so most
Floquet states have a displaced buffer, spread over many $n_b$, and do not
converge with the truncation. The ones that survive are those for which the
storage cancels the drive on the buffer, $g_2 a^2 \approx -\varepsilon_d/2$ --
the cat states and a few excitations above them. At $(35, 18)$ against
$(40, 20)$ that was about 20 converged quasienergies up to `ALPHA_SQ = 0.25`,
14 at 0.5, 4 at 1, 2 at 2, 1 at 3; the larger `DIMS` is there to raise these
counts (printed below). One drive is not enough for 9 parameters beyond the
weakest ones, so all of `ALPHA_SQ_LIST` are fitted at once (each drive is its
own pair of GPU tasks, so adding drives costs no wall clock):

- **shared**: $\delta_a + \Delta_a$, $\delta_b + \Delta_b$, $g_2$, $K_a$, $K_b$,
  $\chi$ -- independent of $\varepsilon_d$ at second order;
- **per drive**: $\varepsilon_d$, $\varepsilon_2$ (no linear scaling with
  $\alpha^2$ assumed: the fit says whether it holds) and a constant `E0`. At
  `ALPHA_SQ = 0` $\varepsilon_d$ and $\varepsilon_2$ do not enter and are held at 0.

`FIT_MODE = "independent"` drops the sharing: every parameter is fitted at each
drive on its own (one `least_squares` per drive), which shows how the "shared"
ones actually move with $\alpha^2$ -- at the price of needing enough converged
eigenvalues at every drive. Each independent fit starts from the joint fit
(`INDEPENDENT_START = "joint"`; `"rwa"` starts from the second-order values):
from the RWA values alone a single drive can wander off to a wrong pairing.

**Too few eigenvalues is an error.** Every drive needs at least its own
parameters + `MIN_DOF` kept eigenvalues (3 per drive in the joint fit, 7 at
`ALPHA_SQ = 0`; all 9 in the independent one), and the fit as a whole its
parameters + `MIN_DOF`; otherwise the effective-model cell raises, naming the
drives that fall short.

`DIMS_BIG` is the second truncation, used only to tell converged eigenvalues
from truncation artefacts (`Q_TOL`, in GHz).

**Only part of the converged spectrum is fitted.** A few dozen eigenvalues per
drive pin down a handful of parameters. Per drive, the fit keeps the
`KEEP_FRACTION` of the converged states with the lowest
$\langle n_a\rangle + \langle n_b\rangle$ -- the most localized: the cat
manifold and the excitations closest to it, where a low-order model is most
accurate -- at least `N_KEEP_MIN` (all of them if fewer converged), at most
`N_KEEP_MAX`. These act on the client only: changing them needs no cluster rerun.

In [ ]:
ALPHA_SQ_LIST = [1,1.5,2,2.5,3,3.5,4]#,4.5,5,5.5,6]
EPS_P = 0.1

DIMS = (50, 25)  # (n_a, n_b): a (storage) is the first tensor factor
DIMS_BIG = (60, 30)
Q_TOL = 1e-6  # max change of a fitted quasienergy between DIMS and DIMS_BIG (GHz)

# One-period integrator, dq.method.Tsit5 on the worker (the client has no dynamiqs).
RTOL, ATOL, MAX_STEPS = 1e-10, 1e-12, 10**6

# Which of the converged states are fitted, per drive (client-side only).
KEEP_FRACTION = 0.3
N_KEEP_MIN, N_KEEP_MAX = 12, 60

# "joint": delta_a, delta_b, g2, K_a, K_b, chi are one value for all drives (eps_d,
# eps2, E0 per drive). "independent": every parameter at each drive on its own.
FIT_MODE = "independent"
INDEPENDENT_START = "rwa"  # start of the independent fits: "joint" fit or "rwa"
MIN_DOF = 3  # kept eigenvalues needed beyond the parameter count (per drive and in all)

## Propagator and Floquet spectrum, on the cluster

`floquet_spectrum` (GPU task, one per drive and truncation) integrates
$U_I(T)$ with `dq.sepropagator` (the tolerances are explicit because everything
downstream inherits them), applies `output_phase`, diagonalizes (numpy, on the
task's CPUs: jax has no non-Hermitian `eig` on GPU), and returns the Floquet
states in the Fock basis. The quasienergies have to stay clear of the zone edge
$\pm\pi/T$ for the log to be unambiguous.

`converged_states` (CPU task, one per drive) is handed the two spectra of a drive
*by reference*: Ray resolves them on the cluster, so the $N\times N$ eigenvector
matrices never leave it. Each Floquet state of `DIMS` is paired with the state of
`DIMS_BIG` it overlaps most; it is converged if that overlap is above 0.99 and
its quasienergy moved by less than `Q_TOL`. All quasienergies come back, but
eigenvectors only for the converged states.

The coordinator `run_spectra` runs as the job driver and asks for no resources;
the tasks request their own. Everything that needs `floquet_lindblad` imports it
on the GPU worker, which also sends back the model constants (`W_A`, `PHI_A`,
`PHI_B`) so `models/ats.py` stays their single source of truth.

In [ ]:
def floquet_spectrum(n_a, n_b, alpha_sq, eps_p, rtol, atol, max_steps):
    """Quasienergies and Floquet states (Fock basis) of the full ATS, one period.

    GPU task: `floquet_lindblad` and the patched dynamiqs come with RUNTIME_ENV
    and are imported here, on the worker.
    """
    import time

    import jax
    import numpy as np

    import dynamiqs as dq
    from floquet_lindblad.models.ats import (
        PHI_A, PHI_B, W_A, build_ats_hamiltonian_interaction,
    )

    if not any(d.platform == "gpu" for d in jax.devices()):
        raise RuntimeError(f"no GPU visible to jax on this worker: {jax.devices()}")

    H_I, _, _, output_phase, V, T, params = build_ats_hamiltonian_interaction(
        n_a=n_a, n_b=n_b, alpha_sq=alpha_sq, epsilon_p=eps_p, n_periods=1
    )
    T = float(T)  # 2 pi / w_a: two drive periods
    method = dq.method.Tsit5(rtol=rtol, atol=atol, max_steps=max_steps)

    t0 = time.time()
    res = dq.sepropagator(H_I, np.array([0.0, T]), method=method, progress_meter=False)
    U_I = np.asarray(res.final_propagator.to_jax())
    t_prop = time.time() - t0

    # Full one-period map in the eigenbasis V of the static Hamiltonian.
    U = np.asarray(output_phase)[:, None] * U_I
    t0 = time.time()
    lam, w = np.linalg.eig(U)
    t_eig = time.time() - t0
    quasi = -np.angle(lam) / T  # = (i/T) log(lam), principal branch
    vecs = np.asarray(V) @ w  # back to the Fock basis |n_a, n_b>
    vecs = vecs / np.linalg.norm(vecs, axis=0)

    N = n_a * n_b
    return dict(
        dims=(n_a, n_b), alpha_sq=alpha_sq, T=T, quasi=quasi, vecs=vecs,
        params={k: float(v) for k, v in params.items()},
        consts=dict(W_A=float(W_A), PHI_A=float(PHI_A), PHI_B=float(PHI_B)),
        diag=dict(t_prop=t_prop, t_eig=t_eig, nsteps=int(res.infos.nsteps),
                  unitarity=float(np.abs(U.conj().T @ U - np.eye(N)).max()),
                  device=str(jax.devices()[0])),
    )


def converged_states(spec, spec_big, q_tol):
    """The states of `spec` whose quasienergy is converged in `spec_big`.

    CPU task, numpy only. Gets both spectra by reference, sends back every
    quasienergy but the eigenvectors of the converged states only.
    """
    import numpy as np

    (n_a, n_b), (n_a2, n_b2) = spec["dims"], spec_big["dims"]
    idx = (np.arange(n_a)[:, None] * n_b2 + np.arange(n_b)[None, :]).ravel()
    # <v_i | v_big_j>, v_i embedded in the big space (zero outside the small box)
    ov = np.abs(spec["vecs"].conj().T @ spec_big["vecs"][idx]) ** 2
    j = ov.argmax(axis=1)
    shift = np.abs(spec["quasi"] - spec_big["quasi"][j])
    overlap = ov[np.arange(len(j)), j]
    sel = np.where((shift < q_tol) & (overlap > 0.99))[0]

    P = np.abs(spec["vecs"]) ** 2
    na, nb = np.repeat(np.arange(n_a), n_b), np.tile(np.arange(n_b), n_a)
    return dict(
        alpha_sq=spec["alpha_sq"], dims=spec["dims"], dims_big=spec_big["dims"],
        T=spec["T"], params=spec["params"], consts=spec["consts"],
        quasi=spec["quasi"], quasi_big=spec_big["quasi"], shift=shift, overlap=overlap,
        mean_na=na @ P, mean_nb=nb @ P, sel=sel, vecs_sel=spec["vecs"][:, sel],
        diag=dict(small=spec["diag"], big=spec_big["diag"]),
    )


def run_spectra(spectrum_task, select_task, alpha_sq_list, dims, dims_big, q_tol, solve_kw):
    """Coordinator: two GPU tasks per drive, chained into one CPU task per drive.

    The tasks are passed in rather than imported: they are defined in this
    notebook, so they ship by value with this function.
    """
    import ray

    from anb_compute import ray as acr

    refs = {}
    for a2 in alpha_sq_list:
        small, big = (acr.submit(spectrum_task, *d, a2, num_gpus=1, num_cpus=8, **solve_kw)
                      for d in (dims, dims_big))
        refs[acr.submit(select_task, small, big, q_tol, num_cpus=8)] = a2

    out, pending = {}, list(refs)
    while pending:
        ready, pending = ray.wait(pending, num_returns=1)
        a2 = refs[ready[0]]
        try:
            out[a2] = ray.get(ready[0])
        except Exception as exc:  # a GPU or CPU task of this drive failed
            out[a2] = {"alpha_sq": a2, "error": f"{type(exc).__name__}: {exc}"}
            print(f"alpha_sq = {a2:<5g} FAILED: {out[a2]['error']}")
            continue
        print(f"alpha_sq = {a2:<5g} {len(out[a2]['sel'])} converged")
    return out


t0 = time.time()
results = acr.run(
    run_spectra, floquet_spectrum, converged_states,
    ALPHA_SQ_LIST, DIMS, DIMS_BIG, Q_TOL,
    dict(eps_p=EPS_P, rtol=RTOL, atol=ATOL, max_steps=MAX_STEPS),
    runtime_env=RUNTIME_ENV,
    poll_seconds=5,
    # No resources here on purpose: the tasks inside ask for their own.
)
print(f"wall clock, incl. job startup: {time.time() - t0:.1f} s\n")

for a2 in ALPHA_SQ_LIST:
    s = results[a2]
    if "error" in s:
        print(f"alpha_sq = {a2:<5g} FAILED: {s['error']}")
        continue
    for name, d in (("DIMS", s["dims"]), ("DIMS_BIG", s["dims_big"])):
        g = s["diag"]["small" if name == "DIMS" else "big"]
        q = s["quasi"] if name == "DIMS" else s["quasi_big"]
        print(f"alpha_sq = {a2:<5g} {d[0]} x {d[1]}: sepropagator {g['t_prop']:.1f} s, "
              f"{g['nsteps']} steps, eig {g['t_eig']:.1f} s | max|U^dag U - 1| = "
              f"{g['unitarity']:.1e} | quasienergies in [{q.min():.3f}, {q.max():.3f}], "
              f"zone edge +-{np.pi / s['T']:.3f}")

SPECS = {a2: s for a2, s in results.items() if "error" not in s}
if not SPECS:
    raise RuntimeError("every drive failed, nothing to fit")
T = next(iter(SPECS.values()))["T"]
PARAMS = {a2: s["params"] for a2, s in SPECS.items()}
CONSTS = next(iter(SPECS.values()))["consts"]

### Store the spectra

Both truncations' quasienergies at every drive, the convergence data, and the
converged Floquet states (Fock basis, `DIMS`), with the parameters they were
computed at, to `results/` (untracked). The full eigenvector matrices stay on the
cluster.

In [ ]:
OUT = REPO / "results" / f"floquet_spectrum_eps_p{EPS_P:g}_{DIMS[0]}x{DIMS[1]}.npz"
OUT.parent.mkdir(exist_ok=True)
store = dict(alpha_sq=np.array(list(SPECS)), eps_p=EPS_P, T=T, dims=DIMS,
             dims_big=DIMS_BIG, q_tol=Q_TOL)
for i, s in enumerate(SPECS.values()):
    store.update({f"{k}_{i}": s[k] for k in
                  ("quasi", "quasi_big", "shift", "overlap", "mean_na", "mean_nb",
                   "sel", "vecs_sel")})
    store.update({f"param_{k}_{i}": v for k, v in s["params"].items()})
np.savez(OUT, **store)
print(f"wrote {OUT} (index i of quasi_i / vecs_sel_i: position in alpha_sq; "
      "vecs_sel_i[:, j] is the state sel_i[j])")

## Which eigenvalues to fit

Converged (computed on the cluster, above), then the `KEEP_FRACTION` of them
with the lowest $\langle n_a\rangle + \langle n_b\rangle$, within
`[N_KEEP_MIN, N_KEEP_MAX]`. `SEL[a2]` holds the kept states' indices in the
`DIMS` spectrum, `QUASI[a2]` / `VECS[a2]` their quasienergies and Floquet states.
In the plot, filled markers are the kept states.

In [ ]:
SEL, QUASI, VECS, SHIFT, MEAN_NA, MEAN_NB = {}, {}, {}, {}, {}, {}
for a2, s in SPECS.items():
    sel, SHIFT[a2], MEAN_NA[a2], MEAN_NB[a2] = s["sel"], s["shift"], s["mean_na"], s["mean_nb"]
    n_conv = len(sel)
    n_keep = min(n_conv, N_KEEP_MAX, max(N_KEEP_MIN, int(np.ceil(KEEP_FRACTION * n_conv))))
    order = np.argsort(MEAN_NA[a2][sel] + MEAN_NB[a2][sel], kind="stable")[:n_keep]
    SEL[a2], VECS[a2] = sel[order], s["vecs_sel"][:, order]
    QUASI[a2] = s["quasi"][SEL[a2]]
    kept = SEL[a2]
    print(f"alpha_sq = {a2:<5g}: {n_conv:4d} / {len(SHIFT[a2])} quasienergies converged "
          f"to {Q_TOL:.0e} GHz, {n_keep} kept"
          + (f"; kept <n_a> up to {MEAN_NA[a2][kept].max():.2f}, "
             f"<n_b> up to {MEAN_NB[a2][kept].max():.2f}" if n_keep else ""))

cmap = mpl.colormaps["plasma"]
norm = mpl.colors.Normalize(vmin=min(SPECS), vmax=max(SPECS))
fig, ax = plt.subplots(figsize=(5.6, 3.8), constrained_layout=True)
for a2 in SPECS:
    ax.scatter(MEAN_NA[a2], np.maximum(SHIFT[a2], 1e-14), s=8, facecolors="none",
               edgecolors=cmap(norm(a2)), lw=0.5)
    ax.scatter(MEAN_NA[a2][SEL[a2]], np.maximum(SHIFT[a2][SEL[a2]], 1e-14), s=10,
               color=cmap(norm(a2)))
ax.axhline(Q_TOL, color="k", lw=0.8, ls=":")
ax.set_yscale("log")
ax.set_xlabel(r"$\langle n_a \rangle$")
ax.set_ylabel("|quasienergy change| between truncations (GHz)")
fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, label=r"$\alpha^2$")
plt.show()

In [ ]:
# Quasienergies vs drive. Opacity follows the shift between DIMS and DIMS_BIG on a
# log scale: opaque at shift <= Q_TOL, faint at shift >= SHIFT_FAINT. States whose
# overlap with DIMS_BIG is below 0.99 are drawn faint. Black rings mark the states
# kept for the fit.
SHIFT_FAINT = 1e-2  # GHz
ALPHA_MIN = 0.03

fig, ax = plt.subplots(figsize=(6, 4.5), constrained_layout=True)
for a2, s in SPECS.items():
    lo, hi = np.log10(Q_TOL), np.log10(SHIFT_FAINT)
    t = (hi - np.log10(np.maximum(s["shift"], 1e-16))) / (hi - lo)  # 1 at Q_TOL, 0 at SHIFT_FAINT
    alpha = np.clip(t, 0, 1) * (1 - ALPHA_MIN) + ALPHA_MIN
    alpha[s["overlap"] < 0.99] = ALPHA_MIN
    ax.scatter(np.full(len(s["quasi"]), a2), s["quasi"], s=12, color="C0",
               alpha=alpha, lw=0)
    ax.scatter(np.full(len(SEL[a2]), a2), s["quasi"][SEL[a2]], s=30,
               facecolors="none", edgecolors="k", lw=0.6)

ax.scatter([], [], s=30, facecolors="none", edgecolors="k", lw=0.6, label="kept for the fit")
ax.set_xlabel(r"$\alpha^2$")
ax.set_ylabel("quasienergy (GHz)")
ax.set_title(f"opacity: convergence ({Q_TOL:.0e} to {SHIFT_FAINT:.0e} GHz)", fontsize=9)
ax.legend(fontsize=8, frameon=False)
ax.grid(alpha=0.3)
plt.show()


## Effective model

$H_{\rm RWA}$, one operator per parameter, plus a constant `E0` (the identity part
of $\log U$, which $H_{\rm RWA}$ has no term for; it shifts all quasienergies of
one drive alike). `delta_a` and `delta_b` stand for $\delta_a + \Delta_a$ and
$\delta_b + \Delta_b$. The operators are built in numpy (real matrices) at
whatever Fock box is asked for.

Starting point: the second-order RWA values (`sympy_rwa_ats.ipynb`), with
$g_2 = g$ and $\varepsilon_d$ from the builder's `params` at each drive; the
constants (`W_A`, `PHI_A`, `PHI_B`) are those the worker read from `models/ats.py`.

`EXTRA_TERMS` is off by default. Turned on, it adds the photon-number dependence
of the two-photon exchange, $g_{2,a}(a^{\dagger}a^3b^\dagger + {\rm h.c.}) +
g_{2,b}(a^2b^{\dagger2}b + {\rm h.c.})$, which the higher orders of $\sin$ give at
first order in $E_J$ -- a check of what $H_{\rm RWA}$ misses (see the residuals).

**What makes the fit fast.**

- *Fit box.* The kept states are converged, so localized: the model is
  diagonalized in the smallest box `FIT_DIMS` outside of which every kept state
  has less than `TAIL_TOL` weight, plus `FIT_MARGIN`, rather than in all of
  `DIMS`. The fit cell checks the result against the full `DIMS`.
- *Storage parity.* Every term of the model (and of `EXTRA_TERMS`) is even in
  $a$, so it is block-diagonal in $(-1)^{n_a}$: two `eigh` of half the size, about
  4x cheaper. Only the model is split; the Floquet states (the full ATS has odd
  terms in $a$) are matched against both blocks at once.

In [ ]:
EXTRA_TERMS = False
TAIL_TOL = 1e-10  # max weight of a kept state outside the fit box
FIT_MARGIN = (4, 2)  # extra (n_a, n_b) levels beyond that


def ladder(n):
    return np.diag(np.sqrt(np.arange(1.0, n)), 1)


def model_ops(dims):
    """H_RWA term by term in the Fock box `dims` (a first), as real dense matrices."""
    n_a, n_b = dims
    a = np.kron(ladder(n_a), np.eye(n_b))
    b = np.kron(np.eye(n_a), ladder(n_b))
    ad, bd = a.T, b.T
    ops = {
        "delta_a": ad @ a,
        "delta_b": bd @ b,
        "g2": a @ a @ bd + ad @ ad @ b,
        "K_a": -ad @ ad @ a @ a / 2,
        "K_b": -bd @ bd @ b @ b / 2,
        "chi": -ad @ a @ bd @ b,
        "eps_d": (b + bd) / 2,
        "eps2": ad @ ad + a @ a,
        "E0": np.eye(n_a * n_b),
    }
    if EXTRA_TERMS:
        X, Y = ad @ a @ a @ a @ bd, a @ a @ bd @ bd @ b
        ops["g2_a"], ops["g2_b"] = X + X.T, Y + Y.T
    return ops


OP_NAMES = list(model_ops((2, 2)))
if FIT_MODE not in ("joint", "independent"):
    raise ValueError(f"FIT_MODE = {FIT_MODE!r}: 'joint' or 'independent'")
# one value for all drives in the joint fit (second order: they depend on g only)
JOINT_SHARED = [k for k in OP_NAMES if k not in ("eps_d", "eps2", "E0")]
SHARED = JOINT_SHARED if FIT_MODE == "joint" else []
PER_DRIVE = [k for k in OP_NAMES if k not in SHARED]


def theory(params, consts=CONSTS):
    """Second-order RWA values (sympy_rwa_ats.ipynb) at one drive."""
    W_A, PHI_A, PHI_B = consts["W_A"], consts["PHI_A"], consts["PHI_B"]
    g, eps_d = params["g"], params["epsilon_d"]
    return {
        "delta_a": -g**2 / W_A * (100 * PHI_A**4 + 135 * PHI_A**2 * PHI_B**2 + 46 * PHI_B**4)
                   / (15 * PHI_A**2 * PHI_B**2),
        "delta_b": -g**2 / W_A * (135 * PHI_A**4 + 324 * PHI_A**2 * PHI_B**2 + 100 * PHI_B**4)
                   / (30 * PHI_A**4),
        "g2": g,
        "K_a": g**2 / W_A * (40 * PHI_A**2 + 27 * PHI_B**2) / (6 * PHI_B**2),
        "K_b": g**2 / W_A * 2 * PHI_B**2 * (58 * PHI_A**2 + 25 * PHI_B**2) / (15 * PHI_A**4),
        "chi": g**2 / W_A * (135 * PHI_A**2 + 92 * PHI_B**2) / (15 * PHI_A**2),
        "eps_d": eps_d,
        "eps2": -eps_d * g / (8 * W_A),
        "E0": 0.0,
        "g2_a": 0.0,
        "g2_b": 0.0,
    }


THEORY = {a2: theory(PARAMS[a2]) for a2 in SPECS}

DRIVES = list(SPECS)


def make_layout(shared):
    """Parameter vector: (name, None) for a shared parameter, (name, alpha_sq) per drive.

    eps_d and eps2 do not enter the spectrum without drive: held at 0 there.
    """
    per_drive = [k for k in OP_NAMES if k not in shared]
    return [(k, None) for k in shared] + [
        (k, a2) for a2 in DRIVES for k in per_drive
        if not (a2 == 0 and k in ("eps_d", "eps2"))
    ]


LAYOUT = make_layout(SHARED)
JOINT_LAYOUT = make_layout(JOINT_SHARED)


def start_values(layout):
    """Second-order RWA values, in `layout`."""
    return np.array([THEORY[a2 if a2 is not None else DRIVES[0]][k] for k, a2 in layout])


def unpack(x, layout=None):
    """Parameter vector -> {alpha_sq: {name: value}}, held values filled in."""
    layout = LAYOUT if layout is None else layout
    shared = {k: v for (k, a2), v in zip(layout, x) if a2 is None}
    out = {}
    for a2 in DRIVES:
        vals = dict(shared, eps_d=0.0, eps2=0.0)
        vals.update({k: v for (k, d), v in zip(layout, x) if d == a2})
        out[a2] = vals
    return out


# --- enough eigenvalues? ------------------------------------------------------
N_OWN = {a2: sum(d == a2 for _, d in LAYOUT) for a2 in DRIVES}
_short = [a2 for a2 in DRIVES if len(SEL[a2]) < N_OWN[a2] + MIN_DOF]
_n_eig = sum(len(SEL[a2]) for a2 in DRIVES)
if _short or _n_eig < len(LAYOUT) + MIN_DOF:
    raise RuntimeError(
        f"too few eigenvalues for the {FIT_MODE} fit: {_n_eig} kept for {len(LAYOUT)} "
        f"parameters (MIN_DOF = {MIN_DOF})\n"
        + "".join(f"  alpha_sq = {a2:g}: {len(SEL[a2])} kept, {len(SPECS[a2]['sel'])} "
                  f"converged; needs {N_OWN[a2]} parameters + {MIN_DOF}\n" for a2 in _short)
        + "Drop these drives from ALPHA_SQ_LIST, enlarge DIMS or loosen Q_TOL"
        + (", or fit them jointly (FIT_MODE = 'joint')." if FIT_MODE == "independent" else ".")
    )


# --- fit box: where the kept states live ------------------------------------
def tail_weights(vecs, dims):
    """tail[c] = largest weight, over the states, on n >= c -- for n_a and for n_b."""
    P = (np.abs(vecs) ** 2).reshape(*dims, -1)
    return [np.cumsum(m[::-1], axis=0)[::-1].max(axis=1) for m in (P.sum(1), P.sum(0))]


tails = [tail_weights(VECS[a2], DIMS) for a2 in DRIVES]
FIT_DIMS = []
for i, n in enumerate(DIMS):
    worst = np.max([t[i] for t in tails], axis=0)
    small = np.nonzero(worst < TAIL_TOL)[0]
    FIT_DIMS.append(min(n, (small[0] if len(small) else n) + FIT_MARGIN[i]))
FIT_DIMS = tuple(int(n) for n in FIT_DIMS)


def make_problem(box):
    """Model operators and kept Floquet states in the Fock box `box`, split by n_a parity."""
    n_a, n_b = box
    rows = (np.arange(n_a)[:, None] * DIMS[1] + np.arange(n_b)[None, :]).ravel()
    parity = np.repeat(np.arange(n_a), n_b) % 2
    blocks = [np.nonzero(parity == p)[0] for p in (0, 1)]
    ops = model_ops(box)
    return dict(
        box=box,
        ops=[{k: O[np.ix_(ix, ix)] for k, O in ops.items()} for ix in blocks],
        vecs={a2: [VECS[a2][rows][ix] for ix in blocks] for a2 in DRIVES},
    )


PROB = make_problem(FIT_DIMS)

x0 = start_values(LAYOUT)
print(f"{FIT_MODE} fit, {len(LAYOUT)} parameters: shared {SHARED}; per drive {PER_DRIVE} "
      f"at alpha_sq = {DRIVES}")
print(f"fit box {FIT_DIMS} (N = {np.prod(FIT_DIMS)}, parity blocks "
      f"{[len(o['E0']) for o in PROB['ops']]}) instead of {DIMS} (N = {np.prod(DIMS)})")

## Fit

Residual: fitted quasienergy minus model eigenvalue, each Floquet state paired
with one model eigenstate by maximum overlap (`linear_sum_assignment` over both
parity blocks, so no two share one), drive by drive. The pairing is redone at
every evaluation, so it can follow the model; the smallest overlap is printed as
a check that it did not have to jump.

The Jacobian is analytic (Hellmann-Feynman): the model is linear in its
parameters, $H = \sum_k \theta_k O_k$, so
$\partial E_j/\partial\theta_k = \langle w_j|O_k|w_j\rangle$ for the paired
eigenvector $w_j$, from the same diagonalization as the residual -- one per
iteration, instead of one per parameter for finite differences. (It is exact
while the paired eigenvalue is non-degenerate within its parity block.)

With `FIT_MODE = "independent"` there is one `least_squares` per drive, on that
drive's parameters and eigenvalues only (`run_fit` with one block per drive),
started from the joint fit unless `INDEPENDENT_START = "rwa"`.

Parameters are scaled by their starting values (`x_scale`), which span several
decades. The errors are the usual $\sqrt{{\rm diag}((J^TJ)^{-1})\,\chi^2/{\rm dof}}$,
which mean little when the residual is model error rather than noise. At the end,
the fitted model is re-evaluated in the full `DIMS` box, as a check of `FIT_DIMS`.

In [ ]:
def evaluate(x, prob, want_jac=True, drives=None, layout=None):
    """Residuals, their Jacobian (Hellmann-Feynman) and the pairing, at x, in prob["box"].

    Only the drives in `drives` (default: all), with x in `layout` (default: LAYOUT).
    """
    layout = LAYOUT if layout is None else layout
    col = {key: i for i, key in enumerate(layout)}
    r_all, J_all, info = [], [], {}
    for a2, vals in unpack(x, layout).items():
        if drives is not None and a2 not in drives:
            continue
        E, W, ov = [], [], []
        for ops, v in zip(prob["ops"], prob["vecs"][a2]):
            Ep, Wp = np.linalg.eigh(sum(val * ops[k] for k, val in vals.items()))
            E.append(Ep)
            W.append(Wp)
            ov.append(np.abs(v.conj().T @ Wp) ** 2)
        n0 = len(E[0])  # columns [0, n0): even block, [n0, ...): odd block
        ov = np.concatenate(ov, axis=1)
        rows, cols = linear_sum_assignment(-ov)
        info[a2] = dict(overlap=ov[rows, cols], states=SEL[a2][rows])
        r_all.append(QUASI[a2][rows] - np.concatenate(E)[cols])

        J = np.zeros((len(rows), len(layout)))
        if want_jac:
            for p, ops in enumerate(prob["ops"]):
                m = (cols >= n0) == bool(p)
                Wm = W[p][:, cols[m] - p * n0]
                for k in vals:
                    c = col.get((k, a2), col.get((k, None)))
                    if c is not None:  # held parameters have no column
                        J[m, c] = -np.einsum("ij,ij->j", Wm, ops[k] @ Wm)
        J_all.append(J)
    return np.concatenate(r_all), np.concatenate(J_all), info


def run_fit(layout, x_start, blocks):
    """One least_squares per block of drives, on the parameters that block sees.

    Returns the parameters, their errors (per block: sqrt(diag((J^T J)^-1) chi^2/dof)),
    and the evaluation counts and messages.
    """
    x, err = x_start.copy(), np.full(len(x_start), np.nan)
    nfev, njev, messages = 0, 0, set()
    for drives in blocks:
        cols = np.array([i for i, (_, d) in enumerate(layout) if d is None or d in drives])
        memo = {}

        def at(xs):
            """least_squares asks for fun(x) and jac(x) separately: diagonalize once."""
            if memo.get("key") != xs.tobytes():
                xf = x.copy()
                xf[cols] = xs
                r, J, _ = evaluate(xf, PROB, drives=drives, layout=layout)
                memo.update(key=xs.tobytes(), val=(r, J[:, cols]))
            return memo["val"]

        xs0 = x[cols]
        fit = least_squares(lambda xs: at(xs)[0], xs0, jac=lambda xs: at(xs)[1],
                            x_scale=np.maximum(np.abs(xs0), 1e-6), max_nfev=20000)
        x[cols] = fit.x
        dof = len(fit.fun) - len(cols)
        if dof > 0:
            err[cols] = np.sqrt(np.diag(np.linalg.pinv(fit.jac.T @ fit.jac))
                                * (fit.fun @ fit.fun) / dof)
        nfev, njev = nfev + fit.nfev, njev + (fit.njev or 0)
        messages.add(fit.message if fit.success else f"FAILED at alpha_sq = {drives}: {fit.message}")
    return x, err, nfev, njev, messages


r0, _, info0 = evaluate(x0, PROB, want_jac=False)

t0 = time.time()
x_start = x0
if FIT_MODE == "independent" and INDEPENDENT_START == "joint":
    x_joint, *_ = run_fit(JOINT_LAYOUT, start_values(JOINT_LAYOUT), [DRIVES])
    fit_joint = unpack(x_joint, JOINT_LAYOUT)
    x_start = np.array([fit_joint[a2][k] for k, a2 in LAYOUT])
blocks = [DRIVES] if FIT_MODE == "joint" else [[a2] for a2 in DRIVES]
x_fit, err, nfev, njev, messages = run_fit(LAYOUT, x_start, blocks)
r, _, info = evaluate(x_fit, PROB, want_jac=False)
t_fit = time.time() - t0

# the same model in the full DIMS box: did FIT_DIMS cut anything that matters?
r_full, _, info_full = evaluate(x_fit, make_problem(DIMS), want_jac=False)
same_pairs = all(np.array_equal(info[a2]["states"], info_full[a2]["states"]) for a2 in DRIVES)

ERR = {key: e for key, e in zip(LAYOUT, err)}
FIT = unpack(x_fit)
# residuals drive by drive (same order as `evaluate` concatenates them)
bounds = np.cumsum([0] + [len(SEL[a2]) for a2 in DRIVES])
R0 = {a2: r0[bounds[i]:bounds[i + 1]] for i, a2 in enumerate(DRIVES)}
R = {a2: r[bounds[i]:bounds[i + 1]] for i, a2 in enumerate(DRIVES)}

print(f"{FIT_MODE} fit: {len(r)} eigenvalues, {len(x_fit)} parameters, {t_fit:.1f} s, "
      f"{nfev} evaluations, {njev} Jacobians: {'; '.join(sorted(messages))}")
print(f"rms residual: start {np.sqrt(np.mean(r0**2)):.2e}, fit {np.sqrt(np.mean(r**2)):.2e} GHz "
      f"(max {np.abs(r).max():.2e}) | min overlap "
      f"{min(i['overlap'].min() for i in info.values()):.3f}")
print(f"fit box {FIT_DIMS} -> full {DIMS}: max |residual change| "
      f"{np.abs(r_full - r).max():.1e} GHz, pairing "
      + ("unchanged" if same_pairs else "CHANGED (enlarge FIT_MARGIN / lower TAIL_TOL)"))

if FIT_MODE == "joint":
    print(f"\nshared\n{'':>8} {'2nd-order RWA':>14} {'fit':>12} {'+-':>9} {'fit / RWA':>10}")
    th0 = THEORY[DRIVES[0]]
    for k in SHARED:
        v0, v = th0[k], FIT[DRIVES[0]][k]
        ratio = f"{v / v0:10.4f}" if v0 else f"{'':>10}"
        print(f"{k:>8} {v0:14.5e} {v:12.5e} {ERR[(k, None)]:9.1e} {ratio}")

    print(f"\nper drive\n{'alpha_sq':>8} {'n fit':>6} {'rms':>9} {'eps_d RWA':>11} {'eps_d fit':>11} "
          f"{'ratio':>7} {'eps2 RWA':>11} {'eps2 fit':>11} {'+-':>8} {'E0':>10}")
    for a2 in DRIVES:
        th, f = THEORY[a2], FIT[a2]
        rms = np.sqrt(np.mean(R[a2] ** 2))
        if a2 == 0:
            print(f"{a2:8g} {len(SEL[a2]):6d} {rms:9.2e} {'held at 0':>11} {'':>11} {'':>7} "
                  f"{'held at 0':>11} {'':>11} {'':>8} {f['E0']:10.2e}")
        else:
            print(f"{a2:8g} {len(SEL[a2]):6d} {rms:9.2e} {th['eps_d']:11.4e} {f['eps_d']:11.4e} "
                  f"{f['eps_d'] / th['eps_d']:7.4f} {th['eps2']:11.3e} {f['eps2']:11.3e} "
                  f"{ERR[('eps2', a2)]:8.1e} {f['E0']:10.2e}")
else:
    for a2 in DRIVES:
        th, f = THEORY[a2], FIT[a2]
        print(f"\nalpha_sq = {a2:g}: {len(SEL[a2])} eigenvalues, rms "
              f"{np.sqrt(np.mean(R[a2] ** 2)):.2e} GHz\n"
              f"{'':>8} {'2nd-order RWA':>14} {'fit':>12} {'+-':>9} {'fit / RWA':>10}")
        for k in OP_NAMES:
            if (k, a2) not in ERR:
                continue  # held at 0 without drive
            v0, v = th[k], f[k]
            ratio = f"{v / v0:10.4f}" if v0 else f"{'':>10}"
            print(f"{k:>8} {v0:14.5e} {v:12.5e} {ERR[(k, a2)]:9.1e} {ratio}")

## Residuals

Per fitted eigenvalue, against its mean storage occupation, coloured by drive.
A residual that grows with $\langle n_a\rangle$ is a term the model lacks
(photon-number dependent couplings, higher-order Kerr), not noise: the
quasienergies themselves are accurate to ~$10^{-11}$ GHz.

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 3.8), constrained_layout=True)
for a2 in DRIVES:
    ax.scatter(MEAN_NA[a2][info0[a2]["states"]], np.abs(R0[a2]), s=14,
               facecolors="none", edgecolors="0.6")
    ax.scatter(MEAN_NA[a2][info[a2]["states"]], np.abs(R[a2]), s=14, color=cmap(norm(a2)))
ax.scatter([], [], s=14, facecolors="none", edgecolors="0.6", label="2nd-order RWA values")
ax.scatter([], [], s=14, color="k", label="fit")
ax.set_yscale("log")
ax.set_xlabel(r"$\langle n_a \rangle$")
ax.set_ylabel("|quasienergy - model eigenvalue| (GHz)")
ax.legend(fontsize=8, frameon=False)
fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, label=r"$\alpha^2$")
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# Fitted parameters vs alpha_sq, against the second-order RWA. With FIT_MODE = "joint"
# the shared parameters are one number for all drives, hence flat.
LABELS = {
    "delta_a": r"\delta_a+\Delta_a", "delta_b": r"\delta_b+\Delta_b", "g2": "g_2",
    "K_a": "K_a", "K_b": "K_b", "chi": r"\chi", "eps_d": r"\varepsilon_d",
    "eps2": r"\varepsilon_2", "E0": "E_0", "g2_a": "g_{2,a}", "g2_b": "g_{2,b}",
}
w_a = CONSTS["W_A"]
a2s = np.array(DRIVES, dtype=float)

ncols = 3
nrows = -(-len(OP_NAMES) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4.0 * ncols, 3.0 * nrows),
                         constrained_layout=True, squeeze=False)
for ax, k in zip(axes.flat, OP_NAMES):
    fit_k = np.array([FIT[a2][k] for a2 in DRIVES])
    err_k = np.array([ERR.get((k, None) if k in SHARED else (k, a2), np.nan) for a2 in DRIVES])
    rwa_k = np.array([THEORY[a2][k] for a2 in DRIVES])
    ax.plot(a2s, rwa_k / w_a, "s--", color="k", ms=4, lw=0.8, mfc="none", label="2nd-order RWA")
    ax.errorbar(a2s, fit_k / w_a, yerr=err_k / w_a, fmt="o", color="C0", ms=4, capsize=2,
                label="fit")
    ax.set_title(("shared: " if k in SHARED else "per drive: ") + rf"${LABELS[k]}$", fontsize=9)
    ax.set_xlabel(r"$|\alpha|^2$")
    ax.set_ylabel(rf"${LABELS[k]}/\omega_a$")
    ax.grid(alpha=0.3)
for ax in axes.flat[len(OP_NAMES):]:
    ax.set_visible(False)
axes.flat[0].legend(fontsize=8, frameon=False)
fig.suptitle(rf"$\epsilon_p = {EPS_P:g}$, {FIT_MODE} fit", fontsize=10)
plt.show()